# Append 2025-07 to 2026-08 applications to acard_master

Rebuilds the source dataset with the same pipeline as `1. Save the data into table` (credit card history, target, AECB thin file, FTS digital + EMID backfill, demographics) for applications with APPROVEDATE in [2025-07-01, 2026-09-01), applies the E2-B population filters and E3 window logic, aligns the result to the `acard_master` schema, and writes it into `mis_automation.datascience_pbg.acard_master` (only that date range is replaced, so re-runs are idempotent).

In [ ]:
import os, sys
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import NullType

# Locate project root (folder containing config/config.yaml) and import common
_p = os.getcwd()
while not os.path.exists(os.path.join(_p, "config", "config.yaml")):
    _parent = os.path.dirname(_p)
    assert _parent != _p, "config/config.yaml not found above the notebook folder"
    _p = _parent
PROJECT_ROOT = _p
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
import common

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
id_col     = cfg["columns"]["id"]
date_col   = cfg["columns"]["date"]
target_col = cfg["columns"]["target"]
MASTER_TBL = f"{cfg['databricks']['catalog']}.{cfg['databricks']['schema']}.acard_master"

# New extraction window: APPROVEDATE in [NEW_START, NEW_END)
# (extraction key, same as notebook 1; windows are assigned later on cfg date column)
RANGE_COL  = "APPROVEDATE"
NEW_START  = "2025-07-01"
NEW_END    = "2026-09-01"
MOB_MONTHS = 13            # same as DATE_RANGES['full'] in notebook 1
TARGET_WINDOW_MONTHS = 12  # same as notebook 1

# Same column lists as notebook 1
ANALYSIS_COLS = [
    "SNAPSHOT_DATE", "CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE", "APPRECDDATE",
    "DELINQUENCYBUCKET", "CURRENT_STATUS", "REFERENCE_NUMBER", "FTS_Ref_No",
    "EMID_REFERENCENUMBER", "GENERALSTAT",
    "CARD_BUSINESS_SEGMENT", "CUSTOMER_SEGMENT", "VIP", "RULING_FAMILY_FLAG",
    "PRIMARYCARD", "F_SEGMENT_NEW", "SEGMENT_POLICY", "CLASSCODE", "ACTIVE_REPORTING",
]
ID_COLS = [
    "CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE", "APPRECDDATE",
    "REFERENCE_NUMBER", "FTS_Ref_No", "EMID_REFERENCENUMBER", "GENERALSTAT",
    "CARD_BUSINESS_SEGMENT", "CUSTOMER_SEGMENT", "VIP", "RULING_FAMILY_FLAG",
    "PRIMARYCARD", "F_SEGMENT_NEW", "SEGMENT_POLICY", "CLASSCODE", "ACTIVE_REPORTING",
]

master_sdf = spark.table(MASTER_TBL)
MASTER_SCHEMA = master_sdf.schema
master_rows_before = master_sdf.count()
rows_in_range_before = master_sdf.filter(
    (F.col(RANGE_COL) >= NEW_START) & (F.col(RANGE_COL) < NEW_END)).count()
window_counts_before = {r["window"]: r["count"] for r in master_sdf.groupBy("window").count().collect()}

print(f"Project root : {PROJECT_ROOT}")
print(f"Date column  : {date_col}  |  id: {id_col}  |  target: {target_col}")
print(f"Master table : {MASTER_TBL}  ({master_rows_before:,} rows, {len(MASTER_SCHEMA)} cols)")
print(f"Master rows with {RANGE_COL} in [{NEW_START}, {NEW_END}): {rows_in_range_before:,}")
print(f"Master {RANGE_COL} max: {master_sdf.agg(F.max(RANGE_COL)).first()[0]}")
print(f"Config windows: {cfg['windows']}")

In [ ]:
# Same logic as cell 3 of "1. Save the data into table". Only change: the APPROVEDATE
# bounds are [date_start, date_end) so the new range starts exactly where the
# existing master (APPROVEDATE < 2025-07-01) ends, with no gap or overlap.

def load_and_clean_credit_card_data(date_start, date_end, mob_months=37):
    df = spark.sql(f"""
        SELECT *
        FROM mis_automation.silver_credit_card.vw_credit_card_history
        WHERE SNAPSHOT_DATE >= APPROVEDATE
          AND SNAPSHOT_DATE < add_months(APPROVEDATE, {mob_months})
          AND APPROVEDATE >= '{date_start}'
          AND APPROVEDATE < '{date_end}'
          AND UPPER(TRIM(CARD_BUSINESS_SEGMENT)) IN ('CARDS')
          AND UPPER(TRIM(CUSTOMER_SEGMENT)) = 'SALARIED'
    """)
    # Exclusion/inclusion filters are skipped here (as in notebook 1);
    # they are applied later by common.apply_population_filters.
    return df


def generate_target(df, window_months=TARGET_WINDOW_MONTHS):
    """-2 immature, 0 good, -1 indeterminate (bucket 1-3), 1 bad (bucket >= 4)."""
    df = df.copy()
    df["SNAPSHOT_DATE"] = pd.to_datetime(df["SNAPSHOT_DATE"])
    df["APPROVEDATE"]   = pd.to_datetime(df["APPROVEDATE"])
    mask = (
        (df["SNAPSHOT_DATE"] >= df["APPROVEDATE"])
        & (df["SNAPSHOT_DATE"] < df["APPROVEDATE"] + pd.DateOffset(months=window_months))
    )
    window_df = df.loc[mask].copy()
    window_df["_snap_month"] = window_df["SNAPSHOT_DATE"].dt.to_period("M")
    agg = (
        window_df.groupby("CREDITCARDNO")
        .agg(max_bucket=("DELINQUENCYBUCKET", "max"),
             unique_months=("_snap_month", "nunique"))
        .reset_index()
    )
    conditions = [
        agg["unique_months"] < window_months,
        agg["max_bucket"] >= 4,
        agg["max_bucket"] == 0,
        agg["max_bucket"].between(1, 3),
    ]
    agg["target"] = np.select(conditions, [-2, 1, 0, -1], default=np.nan)
    if agg["target"].isna().any():
        print(f"WARNING: {agg['target'].isna().sum()} rows matched no target condition - dropped.")
        agg = agg.dropna(subset=["target"])
    agg["target"] = agg["target"].astype(int)

    id_df = df[ID_COLS].drop_duplicates(subset="CREDITCARDNO")
    result = id_df.merge(agg[["CREDITCARDNO", "target"]], on="CREDITCARDNO", how="inner")
    result["REFERENCE_NUMBER"] = (
        result["REFERENCE_NUMBER"].astype(str).str.replace(r'\.0$', '', regex=True)
    )
    return result


def load_aecb_feature_store(df_target):
    ref_pd = (df_target[["REFERENCE_NUMBER"]].drop_duplicates()
              .rename(columns={"REFERENCE_NUMBER": "ref_num"}))
    ref_pd["ref_num"] = ref_pd["ref_num"].astype(str)
    spark.createDataFrame(ref_pd).createOrReplaceTempView("_aecb_ref_nums_new")

    aecb_pd = spark.sql("""
        SELECT /*+ BROADCAST(r) */ a.*
        FROM mis_automation.silver_bureauone.slv_credit_card_aecb_feature_store a
        INNER JOIN _aecb_ref_nums_new r
               ON CAST(a.Reference_Number AS STRING) = r.ref_num
    """).toPandas()

    skip_cols = {"Reference_Number"}
    aecb_pd = aecb_pd.rename(columns={c: f"aecb_{c}" for c in aecb_pd.columns if c not in skip_cols})
    aecb_pd["Reference_Number"] = aecb_pd["Reference_Number"].astype(str)
    df_target["REFERENCE_NUMBER"] = df_target["REFERENCE_NUMBER"].astype(str)

    aecb_merged = aecb_pd.merge(
        df_target[ID_COLS + ["target"]].drop_duplicates(),
        left_on="Reference_Number", right_on="REFERENCE_NUMBER", how="inner",
    )
    df_thin = aecb_merged.loc[aecb_merged["aecb_file_type_revised"] == "Thin File"].reset_index(drop=True)

    print(f"AECB shape   : {aecb_pd.shape}")
    print(f"Merged shape : {aecb_merged.shape}")
    print(f"Thin-file    : {df_thin.shape[0]:,} rows  |  {df_thin['CREDITCARDNO'].nunique():,} cards")
    print(f"\nTarget distribution (all file types):\n{aecb_merged['target'].value_counts(dropna=False).sort_index()}")
    return aecb_merged, df_thin


def load_fts_features(df_thin_score, journey="digital"):
    if journey != "digital":
        raise ValueError(f"Unknown journey '{journey}'. Supported: 'digital'.")
    fts_features_sdf = spark.table("mis_automation.silver_ftsconnect.fts_features")
    df_thin_score_sdf = spark.createDataFrame(df_thin_score)
    joined_sdf = df_thin_score_sdf.join(
        fts_features_sdf,
        df_thin_score_sdf["FTS_Ref_No"] == fts_features_sdf["fts_ref_no"],
        how="left",
    ).drop(fts_features_sdf["fts_ref_no"])

    existing_cols = set(df_thin_score.columns)
    df_merged = joined_sdf.toPandas()
    new_cols = [c for c in df_merged.columns if c not in existing_cols]
    print(f"Merged shape : {df_merged.shape}")
    if new_cols:
        print(f"Match rate   : {df_merged[new_cols[0]].notna().mean():.2%}  ({len(new_cols)} FTS columns)")
    return df_merged


def load_fts_features_physical(df_thin_score):
    """EMID fallback for rows without FTS_Ref_No (same rules as notebook 1)."""
    if "CREDITCARDNO" not in df_thin_score.columns and df_thin_score.index.name == "CREDITCARDNO":
        df_thin_score = df_thin_score.reset_index()

    no_fts_mask = (
        df_thin_score["FTS_Ref_No"].isna()
        | df_thin_score["FTS_Ref_No"].astype(str).isin(["None", "", "nan"])
    )
    df_no_fts = (df_thin_score.loc[no_fts_mask, ["CREDITCARDNO", "EMID_REFERENCENUMBER", "APPRECDDATE", "APPROVEDATE"]]
                 .drop_duplicates(subset="CREDITCARDNO"))
    df_no_fts = df_no_fts[df_no_fts["EMID_REFERENCENUMBER"].notna()].copy()
    df_no_fts["EMID_REFERENCENUMBER"] = df_no_fts["EMID_REFERENCENUMBER"].astype(str)
    print(f"Cases without FTS_Ref_No (with valid EMID): {len(df_no_fts):,}")
    if df_no_fts.empty:
        return df_thin_score

    lookup_sdf = spark.createDataFrame(df_no_fts)
    fts_sdf = spark.table("mis_automation.silver_ftsconnect.fts_features")
    joined = lookup_sdf.join(fts_sdf, lookup_sdf["EMID_REFERENCENUMBER"] == fts_sdf["fts_master_id_number"], how="inner")

    # Same window as notebook 1: [-60d, +30d) around APPRECDDATE and before APPROVEDATE
    joined_filtered = joined.filter(
        (F.col("fts_statement_create_date") >= F.expr("APPRECDDATE - INTERVAL 60 DAYS"))
        & (F.col("fts_statement_create_date") < F.expr("APPRECDDATE + INTERVAL 30 DAYS"))
        & (F.col("fts_statement_create_date") < F.col("APPROVEDATE"))
    )
    w = Window.partitionBy("CREDITCARDNO").orderBy(F.col("fts_statement_create_date").desc())
    fts_emid_pd = (joined_filtered.withColumn("_rn", F.row_number().over(w))
                   .filter(F.col("_rn") == 1)
                   .drop("_rn", "EMID_REFERENCENUMBER", "APPRECDDATE")
                   .toPandas())
    print(f"Matched via EMID: {fts_emid_pd['CREDITCARDNO'].nunique():,} cards")

    fts_cols = [c for c in fts_emid_pd.columns
                if c.startswith("fts_") and c != "fts_master_id_number" and c in df_thin_score.columns]
    fts_lookup = fts_emid_pd.drop_duplicates(subset="CREDITCARDNO").set_index("CREDITCARDNO")[fts_cols]
    fill_mask = no_fts_mask & df_thin_score["CREDITCARDNO"].isin(set(fts_lookup.index))
    print(f"Rows to backfill: {fill_mask.sum():,}  |  FTS columns: {len(fts_cols)}")
    for col_name in fts_cols:
        df_thin_score.loc[fill_mask, col_name] = (
            df_thin_score.loc[fill_mask, "CREDITCARDNO"].map(fts_lookup[col_name].to_dict())
        )
    if fts_cols:
        n = len(df_thin_score)
        after = df_thin_score[fts_cols[0]].notna().sum()
        print(f"FTS coverage after EMID backfill: {after:,} / {n:,} ({after / n:.1%})")
    return df_thin_score

In [ ]:
# 1. Credit card history for the new approval window
df_spark = load_and_clean_credit_card_data(NEW_START, NEW_END, MOB_MONTHS)
print(f"Raw snapshot rows: {df_spark.count():,}")

# 2. Target labels (12-month window)
df_target = generate_target(df_spark.select(*ANALYSIS_COLS).toPandas())
print(f"\nTarget shape: {df_target.shape}")
print(df_target["target"].value_counts().sort_index())

# 3. AECB feature store (thin file)
aecb_merged, df_thin = load_aecb_feature_store(df_target)

# 4-5. FTS features: digital journey, then EMID backfill
df_thin = load_fts_features(df_thin, journey="digital")
df_thin = load_fts_features_physical(df_thin)

# 6. Drop case-insensitive duplicate columns (keeps first, e.g. Reference_Number)
dupes = df_thin.columns[df_thin.columns.str.lower().duplicated(keep="first")]
if len(dupes):
    print(f"Dropping duplicate columns: {list(dupes)}")
    df_thin = df_thin.drop(columns=dupes)

sdf_new = spark.createDataFrame(df_thin)

# 7. Drop NullType (void) columns; they are re-added with master types in the alignment step
void_cols = [f.name for f in sdf_new.schema.fields if isinstance(f.dataType, NullType)]
if void_cols:
    print(f"Dropping NullType (void) columns: {void_cols}")
    sdf_new = sdf_new.drop(*void_cols)
print(f"\nNew source dataset: {len(df_thin):,} rows, {len(sdf_new.columns)} cols")

In [ ]:
drop_cols = ["DATE_OF_BIRTH", "AGE", "NATIONALITY", "SEX"]
sdf_base = sdf_new.drop(*[c for c in drop_cols if c in sdf_new.columns])
sdf_base.createOrReplaceTempView("_acard_cc_base_new")

sdf_src_new = spark.sql("""
    SELECT
        a.*,
        c.NATIONALITY,
        c.DATE_OF_BIRTH,
        c.SEX,
        FLOOR(MONTHS_BETWEEN(a.APPROVEDATE, c.DATE_OF_BIRTH) / 12) AS AGE
    FROM _acard_cc_base_new a
    LEFT JOIN (
        SELECT DISTINCT Customer_number, NATIONALITY, DATE_OF_BIRTH, SEX
        FROM mis_automation.silver_fin_db.customer_information
        WHERE Customer_number IS NOT NULL
    ) c
    ON a.CIF = c.Customer_number
""")

raw_new = sdf_src_new.toPandas()
print(f"Rows after demographic join: {len(raw_new):,} (before: {len(df_thin):,})")
print(f"AGE missing rate: {raw_new['AGE'].isna().mean():.2%}")

In [ ]:
# Validate target (spec 2.2)
assert raw_new[target_col].isna().sum() == 0, "target has nulls - STOP"
assert set(raw_new[target_col].unique()) <= {1, 0, -1, -2}, "unexpected target values - STOP"

# Exact duplicates on id + card (E2-B rule)
n_before = len(raw_new)
raw_new = raw_new.drop_duplicates(subset=[id_col, "CREDITCARDNO"], keep="first").reset_index(drop=True)
print(f"Dropped {n_before - len(raw_new)} duplicate {id_col}+CREDITCARDNO rows -> {len(raw_new):,}")

# File type (Appendix G): keep THIN File only
ft_norm = common._norm(raw_new[cfg["file_type"]["source_column"]])
print(f"File type: {ft_norm.value_counts(dropna=False).to_dict()}")

# sample_status / exclusion_reason (spec 2.3)
new_master, waterfall_new = common.apply_population_filters(raw_new, cfg)
display(waterfall_new)

# window / app_month / app_quarter (spec 5.1) + OUT_OF_SCOPE + oot_half (E3)
new_master = common.assign_window(new_master, cfg)
new_master["window"] = new_master["window"].fillna("OUT_OF_SCOPE")
new_master["oot_half"] = pd.NA
d = pd.to_datetime(new_master[date_col])
for half_name, (s, e) in cfg.get("oot_report_splits", {}).items():
    m = (d >= s) & (d < e) & (new_master["window"] == "OOT")
    new_master.loc[m, "oot_half"] = half_name
for c in ["app_month", "app_quarter"]:
    new_master[c] = new_master[c].astype(str)

print("\nWindow x sample_status:")
display(new_master.groupby(["window", "sample_status"]).size().unstack(fill_value=0).reset_index())

In [ ]:
new_sdf = spark.createDataFrame(new_master)

master_cols = [f.name for f in MASTER_SCHEMA.fields]
new_cols    = set(new_sdf.columns)
missing_in_new = [c for c in master_cols if c not in new_cols]
extra_in_new   = sorted(new_cols - set(master_cols))
print(f"Master columns missing in new data (filled with NULL): {missing_in_new}")
print(f"Extra columns in new data (dropped): {extra_in_new}")

# Critical columns must exist in the new data
critical = [id_col, "CREDITCARDNO", date_col, target_col, "sample_status", "exclusion_reason",
            "window", "app_month", "app_quarter", cfg["file_type"]["source_column"]]
assert not [c for c in critical if c in missing_in_new], f"Critical columns missing: {[c for c in critical if c in missing_in_new]}"

from pyspark.sql.types import StructType, StructField, DecimalType

common_cols = [c for c in master_cols if c in new_cols]
nn_before = new_sdf.select([F.count(F.col(c)).alias(c) for c in common_cols]).first().asDict()
new_types = {f.name: f.dataType for f in new_sdf.schema.fields}

def align(schema):
    """Same column order as master; cast to the given schema; missing columns -> typed NULL."""
    return new_sdf.select([
        (F.col(f.name).cast(f.dataType) if f.name in new_cols else F.lit(None).cast(f.dataType)).alias(f.name)
        for f in schema.fields
    ])

def cast_losses(df):
    nn_after = df.select([F.count(F.col(c)).alias(c) for c in common_cols]).first().asDict()
    return {c: nn_before[c] - nn_after[c] for c in common_cols if nn_before[c] != nn_after[c]}

# 1st pass: cast to the exact master schema
cast_loss = cast_losses(align(MASTER_SCHEMA))
print(f"Columns losing values when cast to master types: {cast_loss}")

# Widen ONLY decimal columns whose new values overflow the master precision (lossless for old rows)
widened = {}
fields = []
for f in MASTER_SCHEMA.fields:
    t = f.dataType
    if f.name in cast_loss and isinstance(t, DecimalType) and isinstance(new_types.get(f.name), DecimalType):
        n = new_types[f.name]
        scale = max(t.scale, n.scale)
        prec = min(38, max(t.precision - t.scale, n.precision - n.scale) + scale)
        widened[f.name] = (t.simpleString(), f"decimal({prec},{scale})")
        t = DecimalType(prec, scale)
    fields.append(StructField(f.name, t, f.nullable))
TARGET_SCHEMA = StructType(fields)
print(f"Decimal columns widened: {widened}")

aligned = align(TARGET_SCHEMA)
remaining_loss = cast_losses(aligned)
assert not remaining_loss, f"Type cast still creates NULLs (non-decimal or unfixable): {remaining_loss}"
assert [f.name for f in aligned.schema.fields] == master_cols, "Column order mismatch"
print(f"Aligned: {aligned.count():,} rows, {len(aligned.columns)} cols, "
      f"{len(MASTER_SCHEMA) - len(widened)} column types identical to master, {len(widened)} widened")

In [ ]:
# Cards already in master with an earlier APPROVEDATE are the same application (same CRNNO and
# APPRECDDATE) whose approval date was restated in the source view -> keep the master row, drop here.
out_of_range = ~((F.col(RANGE_COL) >= NEW_START) & (F.col(RANGE_COL) < NEW_END))
existing_cards = master_sdf.filter(out_of_range).select("CREDITCARDNO").distinct()
dup = aligned.join(existing_cards, "CREDITCARDNO", "left_semi")
n_dup = dup.count()
print(f"Cards already in master with an earlier {RANGE_COL} (dropped from new batch): {n_dup}")
if n_dup:
    old_v = master_sdf.filter(out_of_range).select(
        "CREDITCARDNO", *[F.col(c).alias(f"{c}_old") for c in [id_col, "APPROVEDATE", "APPRECDDATE", "window", "sample_status"]])
    display(dup.select("CREDITCARDNO", *[F.col(c).alias(f"{c}_new") for c in [id_col, "APPROVEDATE", "APPRECDDATE", "window", "sample_status"]])
            .join(old_v, "CREDITCARDNO").orderBy("CREDITCARDNO"))
    same_app = dup.join(master_sdf.filter(out_of_range).select("CREDITCARDNO", id_col, date_col), ["CREDITCARDNO", id_col, date_col]).count()
    assert same_app == n_dup, "Some overlapping cards are not the same application (CRNNO/APPRECDDATE differ) - investigate"
aligned = aligned.join(existing_cards, "CREDITCARDNO", "left_anti").select(*master_cols)  # keep master column order
assert aligned.columns == master_cols, "Column order mismatch after de-dup"

chk = aligned.agg(
    F.min(RANGE_COL).alias("min_date"), F.max(RANGE_COL).alias("max_date"),
    F.count("*").alias("n"), F.countDistinct("CREDITCARDNO").alias("n_cards"),
).first()
print(f"New rows: {chk.n:,}  |  cards: {chk.n_cards:,}  |  {RANGE_COL}: {chk.min_date} -> {chk.max_date}")
assert chk.n > 0, "No new rows"
assert str(chk.min_date) >= NEW_START and str(chk.max_date) < NEW_END, "Rows outside the new date range"

# No card overlap left with the rows of master that are outside the replaced range
overlap = aligned.join(existing_cards, "CREDITCARDNO", "left_semi").count()
assert overlap == 0, "Some CREDITCARDNO already exist in master"

# Windows are assigned on the config date column (APPRECDDATE), so a few cards approved
# after NEW_START but received earlier fall into DEV / OOT / OUT_OF_SCOPE
print(f"\nWindow x sample_status of new rows (window on {date_col}):")
aligned.groupBy("window").pivot("sample_status").count().orderBy("window").show()

In [ ]:
# Rows of the new date range are replaced, everything else is kept, so re-running
# the notebook never duplicates rows. Today that range is empty -> effectively an append.
replace_cond = f"{RANGE_COL} >= '{NEW_START}' AND {RANGE_COL} < '{NEW_END}'"
version_before = spark.sql(f"DESCRIBE HISTORY {MASTER_TBL} LIMIT 1").first()["version"]

if not widened:
    # Schema unchanged: only touch the new date range
    (aligned.write.format("delta").mode("overwrite")
        .option("replaceWhere", replace_cond).saveAsTable(MASTER_TBL))
    print(f"replaceWhere write: {replace_cond}")
else:
    # Some decimals must be widened: rewrite the table = existing rows outside the range
    # (cast losslessly to the wider decimals) + new rows
    existing = master_sdf.filter(f"NOT ({replace_cond})").select(
        [F.col(f.name).cast(f.dataType).alias(f.name) for f in TARGET_SCHEMA.fields])
    (existing.unionByName(aligned).write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(MASTER_TBL))
    print(f"Full rewrite with widened decimals {list(widened)}; rollback: RESTORE TABLE {MASTER_TBL} TO VERSION AS OF {version_before}")
display(spark.sql(f"DESCRIBE HISTORY {MASTER_TBL}").select("version", "timestamp", "operation", "operationMetrics").limit(3))

In [ ]:
spark.catalog.refreshTable(MASTER_TBL)
after = spark.table(MASTER_TBL)
n_after = after.count()
n_new_in_tbl = after.filter((F.col(RANGE_COL) >= NEW_START) & (F.col(RANGE_COL) < NEW_END)).count()
n_old_in_tbl = after.filter(~((F.col(RANGE_COL) >= NEW_START) & (F.col(RANGE_COL) < NEW_END))).count()
# Do not re-evaluate `aligned`/`master_sdf` here: their plans hold the pre-write schema and fail after widening
in_range = (F.col(RANGE_COL) >= NEW_START) & (F.col(RANGE_COL) < NEW_END)
old_win = {r["window"]: r["count"] for r in after.filter(~in_range).groupBy("window").count().collect()}
new_win = {r["window"]: r["count"] for r in after.filter(in_range).groupBy("window").count().collect()}
print(f"New rows per window: {new_win}")

results = [
    ("Columns/order unchanged, only listed decimals widened",
        [f.name for f in after.schema.fields] == master_cols
        and all(a.dataType == m.dataType for a, m in zip(after.schema.fields, MASTER_SCHEMA.fields) if a.name not in widened)),
    ("Existing rows untouched", n_old_in_tbl == master_rows_before - rows_in_range_before),
    ("Total = old + new", n_after == master_rows_before - rows_in_range_before + chk.n),
    ("New range row count = new dataset", n_new_in_tbl == chk.n),
    ("No duplicate CRNNO+CREDITCARDNO", after.groupBy(id_col, "CREDITCARDNO").count().filter("count > 1").count() == 0),
    ("No null window / sample_status", after.filter(F.col("window").isNull() | F.col("sample_status").isNull()).count() == 0),
    ("target only {1,0,-1,-2}", after.filter(~F.col(target_col).isin(1, 0, -1, -2) | F.col(target_col).isNull()).count() == 0),
    ("Window counts of existing rows unchanged", old_win == window_counts_before),
    ("New rows per window sum to new dataset", sum(new_win.values()) == chk.n),
]
for name, ok in results:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"\nMaster rows: {master_rows_before:,} -> {n_after:,}  (+{n_after - master_rows_before:,})")
after.groupBy("window", "sample_status").count().orderBy("window", "sample_status").show(50)
assert all(ok for _, ok in results), "Acceptance checks failed"

In [ ]:
%sql
SELECT MAX(APPROVEDATE) FROM mis_automation.datascience_pbg.acard_master